In [ ]:
pip install openai

In [ ]:
import json
import tiktoken # for token counting
import numpy as np
from collections import defaultdict

In [ ]:
pip install pandas pyarrow

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
df_from_parquet = pd.read_parquet('train-00000-of-00086.parquet', engine='pyarrow')

json_string = df_from_parquet.to_json(orient='records', indent=2)

print("Converted JSON:")
print(json_string)

# Save to a file
with open('output_data.json', 'w') as f:
    f.write(json_string)
print("\nJSON saved to 'output_data.json'.")


JSON saved to 'output_data.json'.


In [ ]:
from openai import OpenAI
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

client = OpenAI()

sampled_df = df_from_parquet.sample(frac=0.5, random_state=42) # Using random_state for reproducibility

# Convert the sampled DataFrame to JSONL format (one JSON object per line).
output_jsonl_filename = 'formatted_fine_tuning_data.jsonl'
# with open(output_jsonl_filename, 'w') as f:
#     for _, row in sampled_df.iterrows():
#         json_record = row.to_json(orient='columns') # orient='columns' for consistent JSON object per row
#         f.write(json_record + '\n')

# print(f"\nSampled data (50%) saved to '{output_jsonl_filename}' in JSONL format.")

try:
  response = client.files.create(
    file=open(output_jsonl_filename, "rb"),
    purpose="fine-tune"
  )
  print(f"File uploaded successfully! File ID: {response.id}")
except Exception as e:
  print(f"Error uploading file: {e}")




File uploaded successfully! File ID: file-4dhySjkBYd1EUDwRfL9kJK


In [ ]:
from datasets import load_dataset

ds = load_dataset("taesiri/video-game-question-answering")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/385 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36705 [00:00<?, ? examples/s]

In [ ]:
display(ds.data)

{'train': MemoryMappedTable
 id: string
 image: string
 conversations: string
 model: string
 ----
 id: [["2adf9f84-082b-46dd-9998-943699567bc8","029aaf80-868a-42da-879d-040a8e5fa27c","6a004282-9127-4a5f-a84e-b65c04a2275a","6068a6a0-1dc9-4606-ab01-ffcd567fe43d","1279dd03-6f03-4c32-83b1-332f6254c279",...,"60c4f135-e132-4c2e-83fe-5e9ae77e32e9","1a12438f-99d0-4bad-862b-320023b52583","d3fe210d-2c5f-4c4c-ac01-702874e48c32","8065a776-45ae-424f-974d-d66b02b61e2e","0d63310f-b586-46cb-8fbf-b8382e87dad9"],["d9b75881-b2fa-4988-91bb-6be66a73af9d","99c1b429-2604-4c73-ba1d-5fa46d262884","8c8c8c9a-20c7-447b-8dfe-14537bf0f523","ce1dfa7c-22df-48b0-a058-50310b8b05f3","0e53047e-18a7-43d5-8339-86ef76c65adc",...,"e0ecf931-25de-4380-a570-4e4ed8241ee4","fca0b90e-498c-4560-9d52-0868fdf0514f","2fbb0236-7ce7-4c3f-bc15-d89ca7a6e074","c495fd66-1909-4cc0-bf57-6fb52bb05919","9a74d8bc-8871-443c-bd4c-73acb7b083cb"],...,["4266f105-a7a2-4454-83ce-4d2339a20463","521fb512-4d5d-41fd-b1a4-8876d001f407","f375dbde-6ce8-4e4c-

To prepare the data for fine-tuning `gpt-3.5-turbo`, we need to convert it into a JSONL format where each line is a JSON object. This object should contain a `messages` array, with each message having a `role` (e.g., 'user', 'assistant') and `content` field. Let's inspect an example from the dataset to understand its structure.

In [ ]:
print(ds['train'][0])

{'id': '2adf9f84-082b-46dd-9998-943699567bc8', 'image': 'dvljiQgpCAE_frames000005906.jpg', 'conversations': '[{"from": "human", "value": "What is the level of the character in the game?"}, {"from": "gpt", "value": "The character in the game is at level 22."}]', 'model': 'argilla/notus-7b-v1'}


In [ ]:
import json

def format_example_for_openai(example):
    # Parse the json into a Python list of dictionaries
    conversations_list = json.loads(example['conversations'])

    messages = []
    for conv in conversations_list:
        role = "user" if conv['from'] == "human" else "assistant"
        messages.append({"role": role, "content": conv['value']})

    return {"messages": messages}

# Apply the formatting function to the training dataset.
formatted_train_data = [format_example_for_openai(ds['train'][i]) for i in range(min(1000, len(ds['train'])))]

# Save the formatted data to a JSONL file
output_fine_tuning_jsonl = 'formatted_fine_tuning_data.jsonl'
with open(output_fine_tuning_jsonl, 'w') as f:
    for entry in formatted_train_data:
        f.write(json.dumps(entry) + '\n')

print(f"Formatted data for fine-tuning saved to '{output_fine_tuning_jsonl}'.")

# Display the first formatted example to verify
print("\nFirst formatted example:")
print(formatted_train_data[0])

Formatted data for fine-tuning saved to 'formatted_fine_tuning_data.jsonl'.

First formatted example:
{'messages': [{'role': 'user', 'content': 'What is the level of the character in the game?'}, {'role': 'assistant', 'content': 'The character in the game is at level 22.'}]}


In [ ]:
client.fine_tuning.jobs.create(
  training_file="file-4dhySjkBYd1EUDwRfL9kJK",
  model="gpt-3.5-turbo"
)

FineTuningJob(id='ftjob-dVtVf2SR7v4aGj5jIsEh5Bxc', created_at=1775498007, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-8tPftNJP935iEHIhJ9AmT1ct', result_files=[], seed=1588294948, status='validating_files', trained_tokens=None, training_file='file-4dhySjkBYd1EUDwRfL9kJK', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [ ]:
response = client.responses.create(
    model="ft:gpt-3.5-turbo-0125:personal::DRi396D3",
    input="How do i defeat a stealth enemy."
)

print(response.output_text)

To defeat a stealth enemy, use detection abilities, area-of-effect attacks, or items that reveal hidden enemies. Listening for sound cues and watching for movement can also help locate them.


#Code for API Call

In [5]:
pip install fastapi nest-asyncio pyngrok uvicorn

In [1]:
from openai import OpenAI
from google.colab import userdata
import os

#Set up client and test to make sure it's working.
#Remember to set the private key
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

client = OpenAI()

response = client.responses.create(
    model="ft:gpt-3.5-turbo-0125:personal::DRi396D3",
    input="How do i defeat a stealth enemy."
)

print(response.output_text)

You can use detection abilities, area-of-effect attacks, or items that reveal hidden enemies. Listening for sound cues and watching for movement can also help locate them.


In [2]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from google.colab import userdata
from fastapi import FastAPI
import asyncio
from uvicorn.config import Config
from uvicorn.server import Server

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}


@app.get("/ask/{message}")
async def read_message(message: str):
    response = client.responses.create(
      model="ft:gpt-3.5-turbo-0125:personal::DRi396D3",
      input=message
      )
    return {"message": response.output_text}

ngrok.set_auth_token(userdata.get('YOUR_AUTH_TOKEN'))

ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)
nest_asyncio.apply()

config = Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = Server(config)

# Ensure any previous server tasks are cancelled to avoid conflicts
for task in asyncio.all_tasks():
    if 'uvicorn' in str(task) and not task.done():
        task.cancel()
        print("Cancelled existing uvicorn task.")

# Run the server as a task in the existing event loop
asyncio.create_task(server.serve())

Public URL: https://c38e-34-106-231-7.ngrok-free.app


<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77>>

In [3]:
#Stop the session
ngrok.kill()